In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gridstatus

In [19]:
# Reading AI Datacenter locations, filtering out to get active datacenters
datacenters = pd.read_csv('ai_data_centers_consumption_and_locations.csv')

datacenters = datacenters[(datacenters['Country']=='United States') & (datacenters['Current power (MW)'] > 0)]
datacenters['State'] = datacenters['County'].str.split(', ').str[1]
datacenters['County'] = datacenters['County'].str.split(',').str[0]
datacenters.drop(columns = ['Current H100 equivalents',
                            'Current total capital cost (2025 USD billions)',
                            'Owner', 'Users', 'Selected Sources', 'Calculations sheet',
                            'Project', 'Current chip types', 'All chip types', 'Investors',
                            'Construction companies', 'Energy companies', 'Country','Address'], inplace = True)

datacenters_by_county = datacenters.groupby(['County','State']).agg(
    num_datacenters = ('Current power (MW)', 'count'),
    total_power_consumption_mw = ('Current power (MW)', 'sum')
    )
datacenters_by_county = datacenters_by_county.reset_index()
datacenters_by_county['County'] = datacenters_by_county['County'].str.replace(' County', '').str.strip()
datacenters_by_county

,County,State,num_datacenters,total_power_consumption_mw
0,Ada,ID,1,152.00000
1,Bell,TX,1,152.00000
2,Bexar,TX,3,154.00000
3,Cherokee,NC,1,65.00000
4,Chesterfield,VA,1,82.00000
5,Clark,IN,1,178.00000
6,Clay,MO,1,105.00000
7,Columbia,PA,1,108.00000
8,Cumberland,NJ,1,50.00000
9,Dakota,MN,1,178.00000


In [20]:
# reading service territory and utility data
df_st = pd.read_excel('EIA861_2024/Service_Territory_2024.xlsx', sheet_name='Counties_States')
df_ud = pd.read_excel('EIA861_2024/Utility_Data_2024.xlsx', sheet_name='States', header=1)

# filtering for grid columns, fixing misnamed col
iso_cols_raw = ['CAISO', 'ERCOT', 'PJM', 'MISO', 'SPP.1', 'ISONE', 'NYISO']
df_ud_clean = df_ud[['Utility Number', 'State'] + iso_cols_raw].rename(columns={'SPP.1': 'SPP'})

# merging grid data to counties
df_merged = df_st.merge(df_ud_clean, on=['Utility Number', 'State'], how='left')

# getting iso data from csv per county
isos = ['CAISO', 'ERCOT', 'PJM', 'MISO', 'SPP', 'ISONE', 'NYISO']
def get_iso(row):
    for iso in isos:
        if str(row[iso]).strip() == 'Y': return iso
    return 'BILATERAL'

df_merged['ISO'] = df_merged.apply(get_iso, axis=1)
county_to_grid = df_merged[['State', 'County', 'ISO']].drop_duplicates()

datacenters_by_county['State'] = datacenters_by_county['State'].str.strip()
county_to_grid['State'] = county_to_grid['State'].astype(str).str.strip()
county_to_grid['County'] = county_to_grid['County'].astype(str).str.strip()

# mapping locations to load zones pricing node col, had to look up
load_zone_map = {
    ('TX', 'Ellis'): 'LZ_NORTH', ('TX', 'Denton'): 'LZ_NORTH',
    ('TX', 'Bell'): 'LZ_NORTH', ('TX', 'Taylor'): 'LZ_WEST',
    ('TX', 'Dickens'): 'LZ_WEST', ('TX', 'Bexar'): 'LZ_CPS',
    ('VA', 'Loudoun'): 'DOMINION', ('VA', 'Prince William'): 'DOMINION',
    ('VA', 'Henrico'): 'DOMINION', ('VA', 'Chesterfield'): 'DOMINION',
    ('OH', 'Licking'): 'AEP', ('OH', 'Franklin'): 'AEP', ('OH', 'Fairfield'): 'AEP',
    ('IA', 'Warren'): 'ILLINOIS_HUB', ('IA', 'Pottawattamie'): 'ILLINOIS_HUB'
}
county_to_grid['Pricing_Node'] = county_to_grid.set_index(['State', 'County']).index.map(load_zone_map)

In [21]:
datacenters_by_county = datacenters_by_county.merge(county_to_grid, on=['State', 'County'], how='left')
datacenters_by_county = datacenters_by_county.sort_values('ISO', ascending=False).drop_duplicates(subset=['State', 'County'], keep='first')
datacenters_by_county.sort_values(by=['num_datacenters','total_power_consumption_mw'], ascending = [False, False], inplace = True)


,County,State,num_datacenters,total_power_consumption_mw,ISO,Pricing_Node
35,Licking,OH,3,1108.00000,PJM,AEP
31,Henrico,VA,3,562.00000,PJM,DOMINION
42,Maricopa,AZ,3,449.00000,SPP,NaN
3,Bexar,TX,3,154.00000,ERCOT,LZ_CPS
52,Shelby,TN,2,1286.00000,BILATERAL,NaN
40,Madison,MS,2,512.00000,MISO,NaN
49,Prince William,VA,2,323.00000,PJM,DOMINION
51,Sarpy,NE,2,317.00000,SPP,NaN
24,Ellis,TX,2,180.00000,ERCOT,LZ_NORTH
46,Niagara,NY,2,100.00000,NYISO,NaN
